# **TRADING MULTI-AGENT**

# **5. Deployment to Production via WebApp**

We will use **FastAPI** for the *backend*, **Streamlit** for the *frontend*, and a *secure tunne*l with **Localtunnel**.

**5.1 Installation of Deployment Dependencies**

In [64]:
# Install the frameworks for the App and API
!pip install fastapi uvicorn streamlit pydantic requests

# Install localtunnel globally using Node.js to expose Colab ports
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 75.4 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
added 22 packages in 3s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴

**5.2 Creating the Agent Backend with FastAPI (main.py)**

This **API** will expose **two main endpoint**s: one to query the generated **trading signals** (integrating the caching or database configured in BASE_PATH) and another to simulate **interactive chat** with the **multi-agent system.**

In [65]:
%%writefile main.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import os
import json

app = FastAPI(
    title="AlphaAgent SaaS API",
    description="High-availability multi-agent backend for signal consumption (Basic, Advanced and Portfolio Manager)",
    version="1.5"
)

class ChatRequest(BaseModel):
    message: str

# System persistent base path
BASE_PATH = '/content/drive/MyDrive/Trading_Agent_POCv1'

@app.get("/api/signals/{ticker}")
def get_trading_signal(ticker: str):
    """
    High-fidelity endpoint that reads the 'agent_cache.json' file generated in Sprint 4
    and extracts analytical metrics from all 3 layers of the agent system
    """
    ticker_upper = ticker.upper()
    cache_path = os.path.join(BASE_PATH, "agent_cache.json")

    if os.path.exists(cache_path):
        try:
            with open(cache_path, 'r', encoding='utf-8') as f:
                cache_data = json.load(f)

            # Key construction according to the exact nomenclature of Cell 55
            basic_key = f"basic_analysis_{ticker_upper}"
            advanced_key = f"advanced_analysis_{ticker_upper}"

            basic_raw = cache_data.get(basic_key, None)
            advanced_raw = cache_data.get(advanced_key, None)

            if basic_raw or advanced_raw:
                # 1. Secure parsing of Basic AI Agent data
                basic_parsed = {}
                if basic_raw:
                    try:
                        clean_str = basic_raw.replace('```json', '').replace('```', '').strip()
                        basic_parsed = json.loads(clean_str)
                    except:
                        basic_parsed = {"recommendation": "ERROR", "technical_justification": basic_raw}

                # 2. Secure parsing of Advanced AI Agent data
                advanced_parsed = {}
                if advanced_raw:
                    try:
                        clean_str = advanced_raw.replace('```json', '').replace('```', '').strip()
                        advanced_parsed = json.loads(clean_str)
                    except:
                        advanced_parsed = {"recommendation": "ERROR", "justification": advanced_raw}

                # Extraction of baseline recommendations
                rec_basic = basic_parsed.get("recommendation", "HOLD") if isinstance(basic_parsed, dict) else "HOLD"
                rec_adv = advanced_parsed.get("recommendation", "HOLD") if isinstance(advanced_parsed, dict) else "HOLD"
                just_basic = basic_parsed.get("technical_justification", "Trend analysis completed.") if isinstance(basic_parsed, dict) else str(basic_raw)
                just_adv = advanced_parsed.get("justification", "Calculation and interpretation of RSI, SMA and Volatility completed.") if isinstance(advanced_parsed, dict) else str(advanced_raw)

                # 3. Synthetic logic for the Portfolio Manager Agent based on consensus/discrepancy
                if rec_basic == rec_adv:
                    pm_rec = rec_basic
                    pm_just = f"Absolute consensus of the agent committee. Both sub-agents agree on a {pm_rec} strategy for protection and optimization."
                else:
                    pm_rec = rec_adv  # The advanced agent brings greater analytical depth
                    pm_just = f"A discrepancy is detected between the systems. The Basic Agent suggests {rec_basic}, but the Portfolio Manager prioritizes the {rec_adv} action dictated by the Advanced Agent due to the strength of the technical analysis indicators indexed in DuckDB."

                return {
                    "ticker": ticker_upper,
                    "has_data": True,
                    "basic_agent": {"recommendation": rec_basic, "justification": just_basic},
                    "advanced_agent": {"recommendation": rec_adv, "justification": just_adv},
                    "portfolio_manager": {"recommendation": pm_rec, "justification": pm_just}
                }
        except Exception as e:
            raise HTTPException(status_code=500, detail=f"Internal error reading persistent cache: {str(e)}")

    # Structured response Fallback if the asset has not executed in the main loop
    return {
        "ticker": ticker_upper,
        "has_data": False,
        "basic_agent": {"recommendation": "HOLD", "justification": "Active in market processing queue."},
        "advanced_agent": {"recommendation": "HOLD", "justification": "Pending historical volume indexing in DuckDB."},
        "portfolio_manager": {"recommendation": "HOLD", "justification": f"The asset {ticker_upper} has no records in 'agent_cache.json'. Run your agents in the control block to persist the rulings."}
    }

@app.post("/api/chat")
def agent_chat(request: ChatRequest):
    """
    Conversational endpoint tailored to simulate cross-responses and interpretations
from all three logical layers of your agent infrastructure.
    """
    user_msg = request.message.lower()

    if "apple" in user_msg or "aapl" in user_msg:
        return {"response": "🤖 **Basic AI Agent:** AAPL is showing bullish consolidation above its 20-day SMA.\n\n📊 **Advanced AI Agent:** Overbought alerts detected (RSI near 68).\n\n🧠 **Portfolio Manager Agent:** I recommend maintaining current positions and limiting further purchases until a healthy correction."}
    elif "risk" in user_msg or "volatility" in user_msg:
        return {"response": "🛡️ **Portfolio Manager:** Currently, the thresholds remain under structural control."}
    elif "lista" in user_msg or "tickers" in user_msg:
        return {"response": "📁 The ecosystem currently queries and writes centrally to the Drive Data Lake through the unified `agent_cache.json` file."}
    else:
        return {"response": f"I have received your request: '{request.message}'. The 3 agents (Basic, Advanced and Portfolio Manager) are ready to process this request in the next database synchronization cycle."}

Writing main.py


**5.3 Develop the Interactive Interface with Streamlit (app.py)**

In [66]:
%%writefile app.py
import streamlit as st
import requests
import pandas as pd

st.set_page_config(page_title="AlphaAgent SaaS - Multi-Agent Dashboard", layout="wide", page_icon="📈")

st.title("📈 AlphaAgent — Multi-Agent Trading Platform")
st.caption("Visual Modules for Asset Monitoring, Portfolio Monitoring and Interactive Chat (LLM)")
st.markdown("---")

API_URL = "http://localhost:8000"

# --- SIDEBAR: Dynamic Selection ---
st.sidebar.header("⚙️ Control Panel")

# 1. Definition of the set of 30 assets
AVAILABLE_ASSETS = [
    "AAPL", "MSFT", "AMZN", "GOOGL", "META", "TSLA", "NVDA", "NFLX", "AMD", "INTC",
    "ECOPETROL", "PFBCOLOM", "BVC", "GRUPOSURA", "NUTRESA", "JPM", "BAC", "WMT",
    "PG", "XOM", "CVX", "JNJ", "PFE", "V", "MA", "DIS", "NKE", "HD", "COST", "PEP"
]

# 2. Quick option to select the entire universe of assets
select_all = st.sidebar.checkbox("🌍 Select all 30 assets")

# 3. Ticker list assignment logic
if select_all:
    # If you check the box, all assets will be automatically selected
    tickers_list = AVAILABLE_ASSETS
    st.sidebar.info(f"Selected: All {len(AVAILABLE_ASSETS)} assets.")
else:
    # Otherwise, allow a custom selection using multiselect
    tickers_list = st.sidebar.multiselect(
        "Select the tickers to query:",
        options=AVAILABLE_ASSETS,
        default=["AAPL", "MSFT", "AMZN", "GOOGL"]  # Default initial values
    )

# Keep the chat history initialization under the selection
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

# --- VISUAL MODULES (Updated Sprint 5 Tabs) ---
tab_monitor, tab_risk, tab_chat = st.tabs([
    "🖥️ Multi-Agent Asset Monitor",
    "🛡️ Portfolio Monitoring Panel",
    "💬 Interactive Chat (LLM)"
])

# 1. MULTI-AGENT ASSET MONITOR (Layered breakdown of analysis)
with tab_monitor:
    st.subheader("🔍 Recommendation from each agent")
    st.markdown("Individual recommendations from the **Basic AI Agent**, **Advanced AI Agent**, and final decision from the **Portfolio Manager Agent**:")

    for ticker in tickers_list:
        with st.expander(f"📊 Multi-Agent Technician Report: {ticker}", expanded=True):
            try:
                response = requests.get(f"{API_URL}/api/signals/{ticker}")
                if response.status_code == 200:
                    data = response.json()

                    if data.get("has_data", False):
                        col1, col2, col3 = st.columns(3)

                        with col1:
                            st.markdown("### 🤖 Basic AI Agent")
                            rec = data["basic_agent"]["recommendation"]
                            st.metric(label="Technical Recommendation", value=rec)
                            st.caption(data["basic_agent"]["justification"])

                        with col2:
                            st.markdown("### 📊 Advanced AI Agent")
                            rec_adv = data["advanced_agent"]["recommendation"]
                            st.metric(label="Advanced Indicators", value=rec_adv)
                            st.caption(data["advanced_agent"]["justification"])

                        with col3:
                            st.markdown("### 🧠 Portfolio Manager")
                            rec_pm = data["portfolio_manager"]["recommendation"]

                            # Conditional marker according to the final recommendation
                            if "BUY" in rec_pm.upper():
                                st.success(f"🟢 recommendation: {rec_pm}")
                            elif "SELL" in rec_pm.upper():
                                st.error(f"🔴 recommendation: {rec_pm}")
                            else:
                                st.warning(f"🟡 recommendation: {rec_pm}")

                            st.info(data["portfolio_manager"]["justification"])
                    else:
                        st.warning(f"⚠️ {data['portfolio_manager']['justification']}")

            except Exception as e:
                st.error(f"❌ Critical infrastructure error: Unable to connect to the API backend. Details: {str(e)}")

# 2. PORTFOLIO MONITORING PANEL
with tab_risk:
    st.subheader("🛡️ Trading rules")

    col_r1, col_r2, col_r3 = st.columns(3)
    with col_r1:
        st.metric(label="Suggested Capital Exposure (Max)", value="18.600.000", delta="15% this week")
    with col_r2:
        st.metric(label="Status Control Rules (DuckDB)", value="LOW RISK", delta_color="inverse")
    with col_r3:
        st.metric(label="Recorded Maximum Drawdown", value="1.86%")

    st.markdown("---")
    st.warning("⚠️ **Automated System Rule:** If the Technical Agent detects a sudden drop in transaction volume concurrent with a negative moving average crossover or RSI overbought crossover, the **Portfolio Manager Agent** will force the automatic closure of the position to mitigate losses in the fund.")

# 3. INTERACTIVE CHAT (LLM)
with tab_chat:
    st.subheader("💬 Chat Console with Multi-Agent Ecosystem & LLM")
    st.markdown("Converse directly with AI Multi-Agents to generate technical analyses or perform quick queries on DuckDB:")

    for msg in st.session_state.chat_history:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])

    if user_prompt := st.chat_input("For example: What is the stance of the 3 agents for Apple today?"):
        with st.chat_message("user"):
            st.markdown(user_prompt)
        st.session_state.chat_history.append({"role": "user", "content": user_prompt})

        try:
            res = requests.post(f"{API_URL}/api/chat", json={"message": user_prompt})
            if res.status_code == 200:
                bot_reply = res.json().get("response", "Unanswered.")
            else:
                bot_reply = "Error processing message with multi-agent system."
        except Exception:
            bot_reply = "⚠️ Critical error: The FastAPI backend (port 8000) is not responding."

        with st.chat_message("assistant"):
            st.markdown(bot_reply)
        st.session_state.chat_history.append({"role": "assistant", "content": bot_reply})

Writing app.py


**5.4 Background Launch**

In [ ]:
# 1. Run the Backend (FastAPI) in a secure background
!nohup uvicorn main:app --host 0.0.0.0 --port 8000 > fastapi.log 2>&1 &
# 2. Run the Frontend (Streamlit) in a secure background
!nohup streamlit run app.py --server.port 8501 > streamlit.log 2>&1 &

# 3. Obtain the public IP address of the Colab runtime environment (Password for the tunnel)
print("\n" + "="*60)
print("🔑 IMPORTANT STEP:")
print("Copy the following numeric IP address. When you open the Localtunnel link,")
print("Paste it into the 'Tunnel Password' field to unlock the Streamlit interface.")
print("="*60)
!curl https://ipv4.icanhazip.com
print("="*60 + "\n")

# 4. Create the exposed secure tunnel to open the Dashboard
!lt --port 8501

**create README**

In [ ]:
%%writefile README.md

# Trading Multi-Agent (v1)

An end-to-end Multi-Agent AI Trading Platform designed to extract financial market data, build a structured relational data store using DuckDB, and deploy an intelligent, autonomous AI Multi-Agent powered by Gemini 2.5 Flash to perform technical analysis, query data via tool use (function calling), and generate structured quantitative trading recommendations.

## 📌 Project Architecture & Data Flow

The platform follows a modern **Medallion Architecture** coupled with an **Multi-Agent Layer**:

1. **Bronze Layer (Raw Storage):** Connects to the Yahoo Finance API (`yfinance`) to execute hybrid extraction of both local Colombian Stock Exchange (BVC) assets (e.g., `ECOPETROL.CO`, `ISA.CO`) and major international equities (e.g., `AAPL`, `TSLA`, `NVDA`). Data is safely archived in its raw semi-structured CSV form within a persistent Google Drive Data Lake.
2. **Silver Layer (Structured Relational Store):** Utilizes **DuckDB** to build a high-performance analytical database. Raw CSV index structures are parsed, schema types are enforced, and data is normalized into an idempotent relational model (`dim_assets` and `fact_historical_prices`) using robust `INSERT OR REPLACE` transactions.
3. **Multi-Agent Layer (Intelligence Hub):** Deploys a **Gemini 2.5 Flash** Multi-agent using the modern `google-genai` SDK. Equipped with native Tool Calling (Function Calling), the Multi-agent autonomously determines when and how to generate optimized SQL queries against the DuckDB Silver Layer, analyzes short-term trends, and outputs deterministic financial decisions.

---

## 🛠️ Tech Stack

* **Core AI Engine:** Google GenAI SDK (`google-genai`), `gemini-2.5-flash` (with strict JSON response schemas and system instruction constraints).
* **Database & Analytical Engine:** DuckDB (OLAP-optimized, fast vectorized execution engine, embedded persistence).
* **Data Processing & Engineering:** Pandas, NumPy, Python 3.12.
* **Financial Extraction:** Yahoo Finance API (`yfinance`).
* **Storage / Environment:** Google Colab, Google Drive (Persistent Data Lake Storage).
* **Visualization:** Matplotlib, Seaborn.

---

## 📂 Project Structure & Database Schema

### Relational Model Design

#### 1. `dim_assets` (Dimension Table)
Stores descriptive metadata for all monitored financial tokens.
* `ticker` (VARCHAR, Primary Key) - Cleaned ticker handle.
* `asset_name` (VARCHAR) - Full name of the company or equity.
* `market` (VARCHAR) - Market categorization (`BVC` for Colombian market, `GLOBAL` for international equities).
* `currency` (VARCHAR) - Trading currency (`COP`, `USD`).

#### 2. `fact_historical_prices` (Fact Table)
Stores structured granular pricing history.
* `ticker` (VARCHAR, Primary Key / Foreign Key)
* `date` (DATE, Primary Key) - Trading date calendar indicator.
* `open` (DOUBLE) - Opening price.
* `high` (DOUBLE) - Daily session high.
* `low` (DOUBLE) - Daily session low.
* `close` (DOUBLE) - Final session close price.
* `adj_close` (DOUBLE) - Adjusted close price for dividends/splits.
* `volume` (BIGINT) - Volume of shares traded.
* `ingestion_timestamp` (TIMESTAMP) - Audit tracking indicator for incremental loads.

---

## 🚀 Step-by-Step Setup Guide

### 1. Persistent Storage Setup
Ensure your Google Drive is mounted within Google Colab to preserve extracted raw data and the DuckDB analytical binary file:
```python
from google.colab import drive
import os

drive.mount('/content/drive')
BASE_PATH = "/content/drive/MyDrive/Trading_Agent_POCv1"
os.makedirs(os.path.join(BASE_PATH, "data", "raw"), exist_ok=True)
os.makedirs(os.path.join(BASE_PATH, "db"), exist_ok=True))

Writing README.md


In [68]:
%%writefile README1.md
# 📈 AlphaAgent — Multi-Agent Trading Web Application

[![FastAPI](https://img.shields.io/badge/Backend-FastAPI-009688?style=flat-square&logo=fastapi&logoColor=white)](https://fastapi.tiangolo.com/)
[![Streamlit](https://img.shields.io/badge/Frontend-Streamlit-FF4B4B?style=flat-square&logo=streamlit&logoColor=white)](https://streamlit.io/)
[![Deployment - Render](https://img.shields.io/badge/Deploy-Render-46E3B7?style=flat-square&logo=render&logoColor=white)](https://render.com/)
[![Python 3.11](https://img.shields.io/badge/Python-3.11-3776AB?style=flat-square&logo=python&logoColor=white)](https://www.python.org/)

**AlphaAgent** es una plataforma analítica SaaS orientada al mercado financiero que implementa una arquitectura jerárquica multi-agente de Inteligencia Artificial para el análisis técnico y la toma de decisiones de inversión automatizadas. El sistema simula un Comité Corporativo de Inversión dividiendo la carga cognitiva en capas especializadas (Análisis Básico, Análisis Avanzado y Dirección de Portafolio).

## 🔗 Enlaces del Proyecto

* **Repositorio GitHub:** [multi-agent-trading-web-application-WebApp-](https://github.com/jwanxanqak/multi-agent-trading-web-application-WebApp-/tree/main)
* **Aplicación en Vivo (Frontend):** [AlphaAgent Web Dashboard](https://multi-agent-trading-web-application-c44n.onrender.com/)
* **Servicio de Producción (Backend API):** [AlphaAgent API Core](https://multi-agent-trading-web-application.onrender.com/)

---

## 🏗️ Arquitectura del Sistema y Capas Agénticas

La plataforma descentraliza el flujo analítico mediante tres entidades lógicas pre-procesadas en un almacén de caché optimizado (`agent_cache.json`):

1.  **Basic Analysis Agent:** Evalúa tendencias de corto plazo (ventanas de 15 días), calcula rangos promedio de volatilidad diaria y analiza la correlación entre la acción del precio y las divergencias de volumen transaccional básico.
2.  **Advanced Analysis Agent:** Procesa ventanas temporales extendidas e infiere dinámicamente el comportamiento de indicadores técnicos estandarizados como medias móviles simples (SMA_10, SMA_20) y el Índice de Fuerza Relativa (RSI_14) para identificar zonas de sobrecompra o sobreventa.
3.  **Portfolio Manager Agent (Consolidador):** Actúa como el filtro supervisor. Modera las discrepancias analíticas entre los agentes de soporte e integra políticas corporativas para emitir una recomendación final unificada (`BUY`, `HOLD`, `SELL`) con su debida justificación ejecutiva.

---

## ⚡ Características Clave

* **Arquitectura Desacoplada (Decoupled SaaS):** Backend construido sobre una API REST robusta y asíncrona con FastAPI, y un Frontend interactivo de alta disponibilidad implementado con Streamlit.
* **Análisis Técnico Estructurado:** Extracción automatizada de métricas clave, sugerencias detalladas para la generación de gráficos (series temporales, histogramas de volumen) y reportes automatizados en formato JSON.
* **Control de Riesgo y Simulación de Umbrales:** Módulo frontend dedicado al monitoreo visual de métricas de mitigación de riesgo financiero (ej. umbrales de exposición máxima).
* **Consola de Comité Agéntico (Interactive Chat):** Chat interactivo simulado que permite interrogar dinámicamente el criterio técnico del comité de agentes de IA para activos seleccionados.

---

## 🛠️ Stack Tecnológico

* **Backend:** Python 3.11, FastAPI, Pydantic v2 (Validación de tipos de datos de entrada).
* **Frontend:** Streamlit, Requests (Consumo asíncrono de microservicios).
* **Persistencia Analítica:** Caché estructurado JSON optimizado para la eliminación de redundancias de tokens y latencia de inferencia.
* **DevOps / Infraestructura:** Render Blueprint Spec (`render.yaml`), Uvicorn (ASGI Web Server).

---
## 📁 Estructura del Repositorio

```text
├── agent_cache.json   # Base de conocimiento estructurada con análisis técnicos pre-calculados.
├── app.py             # Frontend interactivo construido en Streamlit (Monitor, Riesgo y Chat).
├── main.py            # Orquestador del Backend (FastAPI Core) con enrutamiento de señales y chat.
├── render.yaml        # Infraestructura como Código (IaC) para el despliegue orquestado en la nube.
└── requirements.txt   # Manifiesto de dependencias con versiones fijadas para producción.

Overwriting README1.md


In [73]:
%%writefile README1.md

# ☁️ Phase 5: WebApp Architecture & Prototypical Deployment

This section covers the implementation of a decoupled microservices architecture designed to serve the Multi-Agent Trading System. The system features a **FastAPI Asynchronous Backend Engine** and a **Streamlit Interactive Dashboard Frontend**, integrated dynamically within a Google Colab ecosystem and securely exposed via a reverse-proxy tunnel (`localtunnel`).

```┌──────────────────────────────────────────────┐
                    │               Google Drive                   │
                    │         (Persistent Data Lake)               │
                    │           └─ agent_cache.json                │
                    └──────────────────────┬───────────────────────┘
                                           │ Read-Only (Poll)
                                           ▼
┌──────────────────────┐        ┌──────────────────────┐        ┌──────────────────────┐
│  Localtunnel Proxy   │        │  FastAPI Core API    │        │ Streamlit UI Engine  │
│  (Public Gateway)    │◄──────►│     (Port 8000)      │◄──────►│     (Port 8501)      │
│                   )  │  HTTP  │  - Signal Endpoints  │  HTTP  │  - Asset Monitor     │
│  [Secured via IP]    │        │  - Conversational LLM│        │  - Risk Management   │
└──────────────────────┘        └──────────────────────┘        └──────────────────────┘

---

## 5.1 Environment Setup & Core Dependencies

To construct the decoupled application layer and bridge local network stacks out of the virtualized environment, we provision Python framework constraints alongside Node.js-based routing networks.

## 5.2 Backend Service Infrastructure (`main.py`)

The backend layer is architected via **FastAPI** to expose high-performance RESTful APIs. It implements strict data validation schemas using `pydantic` and acts as the execution layer for parsing, combining, and orchestrating downstream agent consensus policies.

### Architectural Logic Highlights:
* **Payload Sanitation & Parsing Engine:** Automatically extracts, formats, and sanitizes raw JSON block text (stripping markdown syntax wraps like ` ```json `) cached by previous execution cycles in the Google Drive Data Lake.
* **Agent Consensus & Conflict Resolution Algorithm:** Integrates deterministic logic evaluating the outputs of the **Basic AI Agent** and **Advanced AI Agent**. If an analytical divergence is flagged, the **Portfolio Manager Agent** triggers an override pipeline, favoring the deeper mathematical metrics processed via the DuckDB indexing system.
* **Resilience & Fallback Matrix:** Implements a structured graceful degradation fallback mechanism (`has_data: False`) to catch cache misses or unindexed tickers without throwing breaking HTTP 500 runtime faults.

## 5.3 Interactive Frontend User Interface (`app.py`)

The presentation layer is developed in **Streamlit**, offering an analytical cockpit for algorithmic execution tracking. The view layout uses modern UI modules organized across distinct, componentized tabs.

### Module Configuration:
1. **Dynamic Control Sidebar:** Includes a batch asset processing array containing 30 global and local stock market symbols (`AVAILABLE_ASSETS`). It features a global state macro checkbox (`Select all 30 assets`) that dynamically handles massive lookups without raising race conditions or execution bugs.
2. **Multi-Agent Asset Monitor Tab:** Uses explicit state containers (`st.expander`) and clean async HTTP queries to visually contrast agent reasoning side-by-side using unified `st.metric` indicators and conditional rendering markers (`st.success`, `st.error`, `st.warning`).
3. **Portfolio Monitoring & Risk Rules Tab:** Maps real-time data checkpoints, maximum drawdowns, and structural protection alerts managed down-stream via DuckDB.
4. **Conversational Interactive Console Tab:** Simulates asynchronous agent-to-user chat interaction with stateful history maintenance preserved inside `st.session_state`.

## 5.4 Asynchronous Infrastructure Execution & Network Tunneling

Because Google Colab operates within ephemeral, isolated virtual machines, running network daemons directly requires decoupling standard interactive shell blockades. We handle this via `nohup` (No Hang Up) sub-processes and expose internal ports via a reverse-proxy topology.



Overwriting README1.md
